# FinTrust Digital Bank — Week 1: Understand & Plan
## AnalystLab Africa Experience Lab Internship Programme — Week 1 Assignment

**Project:** FinTrust Financial Intelligence & Digital Banking Support Solution
**Track coverage:** Data Analytics · Data Science · Machine Learning Engineering · Generative AI · Project Management
**Submission:** End of Week 1 (Sunday, 11:59 PM WAT) · Version 1.0

---
### Notebook purpose
This notebook reproduces, in code, the entire Week 1 evidence base for the FinTrust project:
data loading, profiling, data-quality checks, KPI computation, analytical answers,
hypothesis tests, a baseline risk model, and the datasets behind every chart.

**Data disclaimer:** All FinTrust customers, transactions and financial information are synthetic
and created for educational purposes.


## 0. Setup — imports and configuration

In [ ]:
import pandas as pd, numpy as np, json
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 100)
plt.rcParams.update({"font.size": 10})

CUST_PATH = "FinTrust_Customer_Data.xlsx"
TXN_PATH  = "FinTrust_Transaction_Data.xlsx"
DICT_PATH = "FinTrust_Data_Dictionary.xlsx"


## 1. Load the approved FinTrust resources
- `FinTrust_Customer_Data.xlsx` — customer master file
- `FinTrust_Transaction_Data.xlsx` — transaction event file
- `FinTrust_Data_Dictionary.xlsx` — field definitions and modelling-use guidance


In [ ]:
cust = pd.read_excel(CUST_PATH)
txn  = pd.read_excel(TXN_PATH)
dd   = pd.read_excel(DICT_PATH)
print("Customer  :", cust.shape)
print("Transaction:", txn.shape)
print("Dictionary :", dd.shape)
dd.head(12)


## 2. Data Understanding — structure, types and completeness
Records, columns, data types, missing values, duplicates and the primary keys. This is the
evidence for **Part B (Data Understanding)** of the Data Analytics track.


In [ ]:
def profile(df, name, key):
    print("="*70); print(name); print("="*70)
    print("records        :", len(df))
    print("columns        :", df.shape[1])
    print("missing cells  :", int(df.isna().sum().sum()))
    print("duplicate rows :", int(df.duplicated().sum()))
    print("key            :", key, "| unique:", df[key].nunique())
    print("\n-- dtypes --"); print(df.dtypes.to_string())
    print("\n-- missing by column (non-zero) --")
    print(df.isna().sum()[df.isna().sum() > 0].to_string())

profile(cust, "CUSTOMER DATA", "Customer_ID")
profile(txn,  "TRANSACTION DATA", "Transaction_ID")

print("\n-- categorical cardinality --")
for c in cust.columns:
    if cust[c].dtype == object:
        print(f"[customer] {c:26s} n={cust[c].nunique():3d}  {sorted(cust[c].dropna().unique())[:6]}")
for c in txn.columns:
    if txn[c].dtype == object:
        print(f"[txn]      {c:26s} n={txn[c].nunique():3d}  {sorted(txn[c].dropna().unique())[:6]}")


## 3. Relationship between the datasets (join integrity)
The customer file is the **dimension**; the transaction file is the **fact**. They join on `Customer_ID`.
This cell proves the join is clean (no orphans, no childless customers).


In [ ]:
print("customers in dimension table :", cust.Customer_ID.nunique())
print("distinct customers in fact   :", txn.Customer_ID.nunique())
print("orphan transactions          :", len(set(txn.Customer_ID) - set(cust.Customer_ID)))
print("customers with zero txns     :", len(set(cust.Customer_ID) - set(txn.Customer_ID)))
v = txn.Customer_ID.value_counts()
print("transactions/customer        : mean %.2f | min %d | max %d" % (v.mean(), v.min(), v.max()))
print("date coverage                :", txn.Transaction_DateTime.min(), "->", txn.Transaction_DateTime.max())

m = txn.merge(cust, on="Customer_ID", how="left")
print("\nmerged rows                  :", len(m))


## 4. Data-quality issues identified (critical thinking)
Two issues that a careless analyst would miss:

1. **`Transaction.Location` is not the customer's home city.** Both fields carry the same 8 labels,
   but they agree only ~12.7% of the time. `Location` = where the transaction happened;
   `City` = the customer's registered city. Joining them as one field corrupts every geographic insight.
2. **`Customer_Name` is not an identifier.** The data dictionary marks it `Modelling_Use = No`
   ("identification only"). It must never be used as a join key.
3. **Missing cells:** 192 (only `Device_Type` 96 + `Location` 96) — a single missing row-pair pattern.


In [ ]:
print("Location values :", sorted(txn.Location.dropna().unique()))
print("City values     :", sorted(cust.City.unique()))
print("Location == City match rate: %.1f%%" % (100 * (m.Location == m.City).mean()))
print("\nCustomer_Name uniqueness :", cust.Customer_Name.nunique(), "distinct names for", len(cust), "customers")
print("Dictionary says Modelling_Use:")
print(dd[dd.Field_Name.isin(["Customer_Name","Customer_ID"])][["Field_Name","Modelling_Use"]].to_string(index=False))


## 5. Core metrics — the shared evidence base
Computed bank-wide KPIs used by every track.


In [ ]:
amt = txn.Amount_NGN
metrics = {
 "total_transactions": len(txn),
 "total_value_NGN": round(amt.sum(), 2),
 "average_amount_NGN": round(amt.mean(), 2),
 "median_amount_NGN": round(amt.median(), 2),
 "p95_amount_NGN": round(amt.quantile(.95), 2),
 "max_amount_NGN": round(amt.max(), 2),
 "amount_skew": round(amt.skew(), 2),
 "success_rate_%": round(100 * (txn.Transaction_Status == "Successful").mean(), 2),
 "failed_rate_%": round(100 * (txn.Transaction_Status == "Failed").mean(), 2),
 "reversed_rate_%": round(100 * (txn.Transaction_Status == "Reversed").mean(), 2),
 "pending_rate_%": round(100 * (txn.Transaction_Status == "Pending").mean(), 2),
 "risk_review_rate_%": round(100 * (txn.Risk_Review_Flag == "Yes").mean(), 2),
 "value_under_risk_NGN": round(amt[txn.Risk_Review_Flag == "Yes"].sum(), 2),
 "international_share_%": round(100 * (txn.International_Transaction == "Yes").mean(), 2),
 "active_account_rate_%": round(100 * (cust.Account_Status == "Active").mean(), 2),
 "dormant_rate_%": round(100 * (cust.Account_Status == "Dormant").mean(), 2),
 "restricted_rate_%": round(100 * (cust.Account_Status == "Restricted").mean(), 2),
 "avg_age": round(cust.Age.mean(), 1),
 "avg_tenure_months": round(cust.Tenure_Months.mean(), 1),
 "avg_digital_engagement": round(cust.Digital_Engagement_Score.mean(), 1),
}
for k, v in metrics.items():
    print(f"{k:28s} {v}")

pd.Series(metrics).to_csv("kpi_metrics.csv", header=["value"])
print("\nSaved kpi_metrics.csv")


## 6. Analytical questions (Data Analytics track)
Each question is answered with a computed cross-tabulation.


In [ ]:
def risk_by(col):
    g = (m.groupby(col)
           .agg(transactions=("Risk_Review_Flag", "size"),
                risk_yes=("Risk_Review_Flag", lambda s: (s == "Yes").sum()),
                avg_amount=("Amount_NGN", "mean")))
    g["risk_rate_%"] = (100 * g.risk_yes / g.transactions).round(2)
    g["avg_amount"] = g.avg_amount.round(2)
    return g.sort_values("risk_rate_%", ascending=False)

for c in ["Channel", "Transaction_Type", "Transaction_Status", "Device_Type",
          "International_Transaction", "Customer_Segment"]:
    print(f"\n===== {c} ====="); print(risk_by(c).to_string())

print("\n===== Transaction type: average value =====")
print(m.groupby("Transaction_Type").Amount_NGN.agg(["count","mean","sum"]).round(2).sort_values("mean", ascending=False).to_string())

print("\n===== Monthly trend =====")
print(m.groupby(m.Transaction_DateTime.dt.to_period("M"))
       .agg(transactions=("Transaction_ID","size"),
            value_NGN=("Amount_NGN","sum"),
            success_rate=("Transaction_Status", lambda s: round(100*(s=="Successful").mean(),2))).to_string())


## 7. Hypothesis testing (Data Science track)
Formal statistical tests for the four hypotheses stated in the Week 1 plan.


In [ ]:
def chi_square(col):
    tab = pd.crosstab(m[col], m.Risk_Review_Flag)
    x2, p, dof, _ = stats.chi2_contingency(tab)
    return round(x2,2), dof, p

print("H1  International transactions are more likely flagged")
print("   ", chi_square("International_Transaction"), "->", risk_by("International_Transaction")["risk_rate_%"].to_dict())

print("\nH2  Higher-value transactions are more likely flagged")
a_yes = m.Amount_NGN[m.Risk_Review_Flag == "Yes"]; a_no = m.Amount_NGN[m.Risk_Review_Flag == "No"]
t, p = stats.ttest_ind(a_yes, a_no, equal_var=False)
print(f"    mean risk=Yes NGN {a_yes.mean():,.0f} vs risk=No NGN {a_no.mean():,.0f} | t={t:.2f} p={p:.3e}")

print("\nH3  Failed transactions are more likely flagged")
print("   ", chi_square("Transaction_Status"), "->", risk_by("Transaction_Status")["risk_rate_%"].to_dict())

print("\nH4  Night-time (00:00-05:00) transactions are more likely flagged")
m["Hour"] = m.Transaction_DateTime.dt.hour
m["IsNight"] = m.Hour.isin([0,1,2,3,4,5])
print("   ", chi_square("IsNight"), "->", risk_by("IsNight")["risk_rate_%"].to_dict())


## 8. Baseline predictive model (Data Science track)
Target: `Risk_Review_Flag`. A deliberately simple, fully reproducible baseline so that Week 2
has a number to beat.


In [ ]:
feats = ["Amount_NGN","International_Transaction","Transaction_Status","Transaction_Type",
         "Channel","Device_Type","Customer_Segment","Account_Status","Age","Tenure_Months",
         "Digital_Engagement_Score","Hour"]
X = m[feats].copy()
X["International_Transaction"] = (X.International_Transaction == "Yes").astype(int)
X = pd.get_dummies(X, columns=["Transaction_Status","Transaction_Type","Channel",
                               "Device_Type","Customer_Segment","Account_Status"], drop_first=True)
X = X.fillna(0).astype(float)
y = (m.Risk_Review_Flag == "Yes").astype(int)

print("target positive rate: %.2f%%  (imbalance ratio %.1f:1)" % (100*y.mean(), (1-y.mean())/y.mean()))

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)
num = ["Amount_NGN","Age","Tenure_Months","Digital_Engagement_Score","Hour"]
sc = StandardScaler().fit(Xtr[num])
Xtr[num] = sc.transform(Xtr[num]); Xte[num] = sc.transform(Xte[num])

model = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
proba = model.predict_proba(Xte)[:, 1]

print("\nROC-AUC           : %.3f" % roc_auc_score(yte, proba))
print("majority baseline : %.3f (do-nothing accuracy)" % (1 - yte.mean()))
print("\nconfusion matrix [true 0/1 x pred 0/1]:"); print(confusion_matrix(yte, (proba >= .5).astype(int)))
print(classification_report(yte, (proba >= .5).astype(int), digits=3))

print("\nTop coefficients (standardised):")
print(pd.Series(model.coef_[0], index=X.columns).sort_values(key=abs, ascending=False).head(12).round(3).to_string())


### Honest reading of the baseline
- The model **ranks** transactions correctly (ROC-AUC ≈ 0.67 > 0.50), but at the default 0.5
  threshold it collapses onto the majority class (recall ≈ 0.04).
- **Week 2 fix:** `class_weight='balanced'`, threshold tuning, and gradient-boosted trees.
- Reported accuracy ≈ 0.80 is **not** evidence of skill — it equals the do-nothing baseline.


## 9. Charts and diagrams (submission evidence)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

g = risk_by("Channel"); ax[0,0].bar(g.index, g["risk_rate_%"], color="#c0392b")
ax[0,0].set_title("Risk-review rate by channel (%)"); ax[0,0].tick_params(axis="x", rotation=20)

g = risk_by("Transaction_Type"); ax[0,1].bar(g.index, g["risk_rate_%"], color="#2c7fb8")
ax[0,1].set_title("Risk-review rate by transaction type (%)"); ax[0,1].tick_params(axis="x", rotation=20)

g = risk_by("International_Transaction"); ax[1,0].bar(g.index, g["risk_rate_%"], color="#8e44ad")
ax[1,0].set_title("Risk-review rate: domestic vs international (%)")

ax[1,1].hist(np.log10(m.Amount_NGN), bins=50, color="#16a085")
ax[1,1].set_title("Transaction amount distribution"); ax[1,1].set_xlabel("log10(Amount_NGN)")

plt.tight_layout(); plt.savefig("fig1_eda.png", dpi=150); plt.show()


## 10. Week 2–4 plan (all tracks)

| Week | Deliverable |
|---|---|
| Week 2 | Data Analytics: build the dashboard + SQL queries · Data Science: engineered features + tuned model · MLE: training pipeline skeleton · GenAI: knowledge-base chunking + first prompts · PM: risk register update |
| Week 3 | DS: evaluated model behind the API · MLE: FastAPI `/predict` + tests · DA: deep-dive risk & channel analysis · GenAI: RAG retrieval + evaluation set run |
| Week 4 | Integration, documentation, executive summary, presentation, submission |

---
### Reproduction
Run all cells top to bottom. Data files must sit beside the notebook.
Deterministic: `random_state=42` throughout.
